<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB02 &mdash; Transaction Type Configuration</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Review the default transaction types and configure the lifecycle event types the pack relies on, including the cash-settled option exercise, expiry and future cash settlement movements.</div>
</div>

<sub>IBOR pack sequence: NB00 &nbsp;&rarr;&nbsp; NB01 &nbsp;&rarr;&nbsp; **NB02** &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; NB04 &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07 &nbsp;&rarr;&nbsp; NB08</sub>

# NB02: Transaction Type Configuration

Reviews existing types and configures lifecycle event types: BondCoupon, BondPrincipal, Maturity, SwapCashFlow, TermDeposit events, and futures/options.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import os, json
from datetime import datetime, timedelta, timezone
import pandas as pd
import lusid as lu
import lusid.models as lm
import lumipy as lp
pd.set_option("display.max_columns", None)
SCOPE = 'ibor-training-v8'
QUOTE_SCOPE = 'ibor-training-v8-quotes'

def get_factory(app='lusid'):
    with open("secrets.json") as f: secrets = json.load(f)
    pat = secrets.get("api", {}).get("accessToken")
    module = __import__(app)
    if pat:
        return module.extensions.SyncApiClientFactory(config_loaders=[
            module.extensions.ArgsConfigurationLoader(api_url=secrets["api"]["lusidUrl"], access_token=pat)])
    return module.extensions.SyncApiClientFactory(config_loaders=[
        module.extensions.SecretsFileConfigurationLoader("secrets.json")])

def get_lumi():
    with open("secrets.json") as f: secrets = json.load(f)
    pat = secrets.get("api", {}).get("accessToken")
    if pat:
        return lp.get_client(access_token=pat, api_url=secrets["api"]["lusidUrl"].replace("/api", "/honeycomb"))
    return lp.get_client(api_secrets_file="secrets.json")

lusid_factory = get_factory()
lumi = get_lumi()
print('Ready')
transaction_configuration_api = lusid_factory.build(lu.TransactionConfigurationApi)

---
## Review Existing Transaction Types

In [ ]:
query = "SELECT AliasType, AliasDescription, AliasSource, AliasTransactionClass FROM Lusid.TransactionType WHERE AliasSource = 'default' LIMIT 30"
df = lumi.run(query, quiet=True)
print(f"Default transaction types: {len(df)}")
display(df.head(15))

---
## Configure Lifecycle Event Transaction Types
These define how LUSID processes automatic events from instrument definitions.

In [ ]:
# Create the Notional side definition (required for futures transaction types)
try:
    transaction_configuration_api.set_side_definition(
        side="Notional",
        side_definition_request=lm.SideDefinitionRequest(
            security="Txn:Units",
            currency="Txn:TradeCurrency",
            rate="Txn:TradeToPortfolioRate",
            units="Txn:Units",
            amount="Txn:NotionalAmount"
        )
    )
    print("Created Notional side definition")
except lu.ApiException as e:
    if 'AlreadyExists' in str(e.body):
        print("Notional side already exists")
    else:
        print(f"Side error: {str(e.body)[:300]}")

In [ ]:
lifecycle_types = [
    {"source": "default", "type": "BondCoupon",
     "aliases": [{"type": "BondCoupon", "desc": "Bond coupon event"}],
     "movements": [
         {"name": "Coupon", "movement_types": "CashCommitment", "side": "Side2", "direction": 1},
         {"name": "Carry", "movement_types": "Carry", "side": "Side1", "direction": 1}]},
    {"source": "default", "type": "BondPrincipal",
     "aliases": [{"type": "BondPrincipal", "desc": "Bond principal return"}],
     "movements": [
         {"name": "Bond Principal", "movement_types": "CashCommitment", "side": "Side2", "direction": 1}]},
    {"source": "default", "type": "Maturity",
     "aliases": [{"type": "Maturity", "desc": "Maturity event"}],
     "movements": [
         {"name": "Maturity", "movement_types": "StockMovement", "side": "Side1", "direction": -1}]},
    {"source": "default", "type": "SwapCashFlow",
     "aliases": [{"type": "SwapCashFlow", "desc": "Swap cashflow event"}],
     "movements": [
         {"name": "SwapCashflow", "movement_types": "CashCommitment", "side": "Side2", "direction": 1},
         {"name": "Carry", "movement_types": "Carry", "side": "Side1", "direction": 1}]},
    {"source": "default", "type": "TermDepositInterest",
     "aliases": [{"type": "TermDepositInterest", "desc": "Term deposit interest"}],
     "movements": [
         {"name": "Interest", "movement_types": "CashCommitment", "side": "Side2", "direction": 1},
         {"name": "Carry", "movement_types": "Carry", "side": "Side1", "direction": 1}]},
    {"source": "default", "type": "TermDepositPrincipal",
     "aliases": [{"type": "TermDepositPrincipal", "desc": "Term deposit principal return"}],
     "movements": [
         {"name": "Principal", "movement_types": "CashCommitment", "side": "Side2", "direction": 1}]},
    {"source": "default", "type": "CashSettledOptionExercise",
     "aliases": [{"type": "CashSettledOptionExercise", "desc": "Cash settled option exercise"}],
     "movements": [
         {"name": "Remove option", "movement_types": "StockMovement", "side": "Side1", "direction": -1},
         {"name": "Cash settlement", "movement_types": "CashCommitment", "side": "Side2", "direction": 1}]},
    {"source": "default", "type": "Expiry",
     "aliases": [{"type": "Expiry", "desc": "Option/future expiry (OTM)"}],
     "movements": [
         {"name": "Remove holding", "movement_types": "StockMovement", "side": "Side1", "direction": -1}]},
    {"source": "default", "type": "FutureCashSettlement",
     "aliases": [{"type": "FutureCashSettlement", "desc": "Futures cash settlement"}],
     "movements": [
         {"name": "Remove notional", "movement_types": "StockMovement", "side": "Notional", "direction": -1},
         {"name": "Realise PnL", "movement_types": "CashReceivable", "side": "Side2", "direction": 1}]},
]

for lt in lifecycle_types:
    try:
        request = lm.TransactionConfigurationDataRequest(
            aliases=[lm.TransactionConfigurationTypeAlias(
                type=a["type"], description=a["desc"],
                transaction_class="Basic", transaction_roles="AllRoles", is_default=False
            ) for a in lt["aliases"]],
            movements=[lm.TransactionConfigurationMovementDataRequest(
                name=m["name"], movement_types=m["movement_types"],
                side=m["side"], direction=m["direction"],
                movement_options=[], mappings=[]
            ) for m in lt["movements"]]
        )
        transaction_configuration_api.set_transaction_type(
            source=lt["source"], type=lt["type"], transaction_type_request=request)
        mvs = ", ".join(f"{m['movement_types']}/{m['side']}" for m in lt["movements"])
        print(f"  {lt['type']}: {mvs}")
    except lu.ApiException as e:
        print(f"  Error on {lt['type']}: {str(json.loads(e.body))[:200]}")

print("\nNB02 Complete. Proceed to NB03: Transaction Loading.")